In [0]:
from pyspark.sql.functions import when, col, pandas_udf
from pyspark.sql.types import BooleanType
import pandas as pd

spark


In [0]:
pdf = pd.DataFrame({
      "site_id": ["Leicester Forest East", "South Mimms", "Cobham Services", "Reading Services", "Oxford Services"],                                                                                                                   
      "carbon_intensity": [210, 185, 310, 95, 260],                                                                                                           
      "temperature": [12.5, 8.3, 15.1, 6.7, 11.2]                                                                                                             
  })

sdf = spark.createDataFrame(pdf)

sdf.show()

In [0]:
sdf.drop('temperature').filter(sdf['carbon_intensity'] > 200).withColumn('high_carbon', when(sdf['carbon_intensity'] > 250, True).otherwise(False)).toPandas()

In [0]:
@pandas_udf(BooleanType())
def is_high_carbon(carbon_intens_col: pd.Series) -> pd.Series:
    return carbon_intens_col > 250

sdf.withColumn('high_carbon', is_high_carbon(sdf['carbon_intensity'])).show()



In [0]:
pdf = pd.DataFrame({
      "site_id": ["Leicester Forest East", "Leicester Forest East", "Leicester Forest East", "South Mimms", "South Mimms", "South Mimms"],
      "carbon_intensity": [210, 185, 310, 95, 260, 180],
      "temperature": [12.5, 8.3, 15.1, 6.7, 11.2, 9.4]
  })

sdf = spark.createDataFrame(pdf)
sdf.show()


In [0]:
def mean_carbon_per_site(site_df: pd.DataFrame) -> pd.DataFrame:
    # site_df is one site's partition — pure pandas
    mean = site_df['carbon_intensity'].mean()
    site = site_df['site_id'].values[0]
    mean_df = pd.DataFrame({"site_id": [site], "mean_carbon": [mean]})
    return mean_df

In [0]:
sdf.groupBy('site_id').applyInPandas(mean_carbon_per_site, schema="site_id string, mean_carbon double").show()  